# Principal Component Analysis (PCA)

The objective of this notebook is to reduce the dimensionality of the engineered feature space while preserving as much information as possible.

The feature engineering stage produced three variables describing different aspects of Bitcoin market behavior:

- Returns
- Volatility
- Momentum

Although these features provide valuable information, some of their variability may overlap. Principal Component Analysis (PCA) transforms the original features into a new set of orthogonal components that capture the dominant patterns present in the data.

These principal components will subsequently be used as inputs for clustering algorithms in order to identify distinct market regimes.

In [1]:
# Load librarys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
%matplotlib inline

# Ruta absoluta a la raíz del proyecto
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from config import *

In [2]:
# Load feature dataset
asset_name = ASSETS[0].lower().replace("-", "_")

feature_path = (f"{PROCESSED_DATA_PATH}/{asset_name}_features.csv")

df = pd.read_csv(feature_path)

df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

df.head()

,Returns,Volatility,Momentum
Date,,,
2018-01-31,0.011359,0.064866,-0.086472
2018-02-01,-0.102783,0.064014,-0.200817
2018-02-02,-0.037052,0.063907,-0.239214
2018-02-03,0.038973,0.064239,-0.288723
2018-02-04,-0.097865,0.060821,-0.286471


------

### Dataset Verification

The final feature dataset contains 3,051 observations and three engineered variables: Returns, Volatility, and Momentum.

All features are stored as numerical variables (`float64`), making them directly suitable for mathematical transformations and machine learning algorithms. Furthermore, no missing values remain after the feature engineering stage, confirming that the dataset is fully prepared for dimensionality reduction.

The resulting feature matrix provides a compact representation of Bitcoin market behavior and serves as the input for Principal Component Analysis (PCA).

In [3]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

Rows: 3051
Columns: 3
<class 'pandas.DataFrame'>
DatetimeIndex: 3051 entries, 2018-01-31 to 2026-06-08
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Returns     3051 non-null   float64
 1   Volatility  3051 non-null   float64
 2   Momentum    3051 non-null   float64
dtypes: float64(3)
memory usage: 95.3 KB


------

## Feature Scaling

Principal Component Analysis is sensitive to the scale of the input variables. Features with larger numerical ranges can dominate the resulting principal components, even if they are not inherently more informative.

To ensure that Returns, Volatility, and Momentum contribute equally to the analysis, all features are standardized using StandardScaler. This transformation centers each feature around zero and scales it to unit variance.

In [4]:
# Use standar scaler to standarize all the variables
from sklearn.preprocessing import StandardScaler

# Define the features
features = [
    "Returns",
    "Volatility",
    "Momentum"
]

# Apply the StandardScaler
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df[features])

In [5]:
# Convert back to DataFrame
scaled_df = pd.DataFrame(
    scaled_features, 
    columns = features, 
    index = df.index
)

scaled_df.head()

,Returns,Volatility,Momentum
Date,,,
2018-01-31,0.308262,2.666907,-0.789169
2018-02-01,-3.139880,2.600338,-1.664864
2018-02-02,-1.154207,2.592011,-1.958916
2018-02-03,1.142445,2.617944,-2.338070
2018-02-04,-2.991292,2.350934,-2.320826


In [6]:
# Validation of scaling
scaled_df.describe()

,Returns,Volatility,Momentum
count,3.051000e+03,3051.000000,3051.000000
mean,5.822212e-18,0.000000,0.000000
std,1.000164e+00,1.000164,1.000164
min,-1.126346e+01,-1.705842,-3.455620
25%,-4.408701e-01,-0.680713,-0.551818
50%,-1.596328e-02,-0.203028,-0.072927
75%,4.197834e-01,0.461372,0.463196
max,5.628243e+00,4.734112,4.917675
